In [1]:
import pathlib
import sys
import json
import importlib
import inspect
from datetime import datetime, timedelta
from time import perf_counter

cwd = pathlib.Path.cwd().resolve()
if (cwd / "Codex").exists():
    sys.path.insert(0, str(cwd / "Codex"))
elif cwd.name.lower() == "codex":
    sys.path.insert(0, str(cwd))

import buildup as _buildup
_buildup = importlib.reload(_buildup)
get_15min_buildup = _buildup.get_15min_buildup
get_intraday_trade_context = getattr(_buildup, "get_intraday_trade_context", None)
SUPPORTS_SECURITY_ID = "security_id" in inspect.signature(get_15min_buildup).parameters
SUPPORTS_TRADE_CONTEXT = callable(get_intraday_trade_context)



In [2]:
config_path = None
# Try Data folder first (parent directory)
data_config = cwd.parent / "Data" / "config.json"
if data_config.exists():
    config_path = data_config
# Fallback to Codex folder
elif (cwd / "Codex" / "config.json").exists():
    config_path = cwd / "Codex" / "config.json"
# Try current directory
elif (cwd / "config.json").exists():
    config_path = cwd / "config.json"

config = {}
if config_path and config_path.exists():
    with config_path.open("r", encoding="utf-8") as f:
        config = json.load(f)

ACCESS_TOKEN = str(config.get("access_token", "YOUR_ACCESS_TOKEN")).strip() or "YOUR_ACCESS_TOKEN"
CLIENT_ID = str(config.get("client_id", "YOUR_CLIENT_ID")).strip() or "YOUR_CLIENT_ID"

INSTRUMENT_NAMES = ["NIFTY 07 APR 23950 CALL", "NIFTY 07 APR 23850 PUT"]

SECURITY_ID_MAP = {}  # Optional: {"NIFTY 07 APR 22100 PUT": "12345"}
EXCHANGE_SEGMENT = "NSE_FNO"
MAX_ROWS = 9
REQUEST_TIMEOUT = 8
COLOR_SIGNAL = True
FETCH_MARKET_CONTEXT = True
CHAIN_WINGS = 3


In [ ]:
def inr(value):
    n = int(round(float(value)))
    sign = "-" if n < 0 else ""
    s = str(abs(n))
    if len(s) <= 3:
        return sign + s
    last3 = s[-3:]
    rest = s[:-3]
    parts = []
    while len(rest) > 2:
        parts.append(rest[-2:])
        rest = rest[:-2]
    if rest:
        parts.append(rest)
    return sign + ",".join(reversed(parts)) + "," + last3


def _sgn(v, eps=1e-9):
    if v > eps:
        return 1
    if v < -eps:
        return -1
    return 0


def _normalize_inst_key(text):
    return " ".join(str(text).upper().replace(" CE ", " CALL ").replace(" PE ", " PUT ").split())


def direction_confidence_1to5m(closes):
    if len(closes) < 2:
        return {"direction": "NA", "confidence": "LOW"}

    c_last = float(closes[-1])
    d1 = c_last - float(closes[-2])
    d3 = c_last - float(closes[max(0, len(closes) - 4)])
    d5 = c_last - float(closes[max(0, len(closes) - 6)])

    base = c_last if c_last else 1.0
    d1_pct = abs(d1 / base * 100.0)
    d5_pct = abs(d5 / base * 100.0)
    if d1_pct < 0.03 and d5_pct < 0.08:
        return {"direction": "SIDE", "confidence": "LOW"}

    vote = 0.5 * _sgn(d1) + 0.3 * _sgn(d3) + 0.2 * _sgn(d5)
    if vote >= 0.6:
        direction = "UP"
    elif vote <= -0.6:
        direction = "DOWN"
    else:
        direction = "SIDE"

    if direction == "SIDE":
        confidence = "LOW"
    elif abs(vote) >= 0.9 and d5_pct >= 0.35:
        confidence = "HIGH"
    elif abs(vote) >= 0.6 and d5_pct >= 0.15:
        confidence = "MED"
    else:
        confidence = "LOW"
    return {"direction": direction, "confidence": confidence}


def build_interval_meta_map(candles_1m):
    buckets = {}
    for c in candles_1m or []:
        ts = c.get("timestamp")
        close = c.get("close")
        if not ts or close is None:
            continue
        dt = datetime.strptime(ts, "%Y-%m-%d %H:%M")
        start = dt.replace(minute=(dt.minute // 15) * 15, second=0, microsecond=0)
        end = start + timedelta(minutes=15)
        key = f"{start:%H:%M} - {end:%H:%M}"
        buckets.setdefault(key, []).append(float(close))
    return {k: direction_confidence_1to5m(v) for k, v in buckets.items()}


def signal_from_confluence(direction, trading_zone):
    if direction == "SIDE" or direction == "NA":
        return "-"
    if direction == "UP" and trading_zone in {"Long Buildup", "Short Covering"}:
        return "LONG"
    if direction == "DOWN" and trading_zone in {"Short Buildup", "Long Unwinding"}:
        return "SHORT"
    return "-"


def color_signal_text(signal):
    text = f"{signal:<9}"
    if not COLOR_SIGNAL:
        return text
    color = {
        "LONG": "\033[92m",
        "SHORT": "\033[91m",
        "-": "\033[93m",
    }.get(signal, "\033[0m")
    return f"{color}{text}\033[0m"


def _build_context(names):
    if not FETCH_MARKET_CONTEXT or not SUPPORTS_TRADE_CONTEXT:
        return {}
    if not names:
        return {}

    try:
        return get_intraday_trade_context(
            access_token=ACCESS_TOKEN,
            client_id=CLIENT_ID,
            instrument_name=names[0],
            exchange_segment=EXCHANGE_SEGMENT,
            timezone_name="Asia/Kolkata",
            strikes_each_side=CHAIN_WINGS,
            request_timeout=max(REQUEST_TIMEOUT, 8),
        )
    except Exception:
        return {}


def built_up():
    if not ACCESS_TOKEN or ACCESS_TOKEN == "YOUR_ACCESS_TOKEN":
        print("Set access_token in Codex/config.json")
        return
    if not CLIENT_ID or CLIENT_ID == "YOUR_CLIENT_ID":
        print("Set client_id in Codex/config.json")
        return

    names = [INSTRUMENT_NAMES] if isinstance(INSTRUMENT_NAMES, str) else INSTRUMENT_NAMES
    context = _build_context(names)
    context_lookup = context.get("instrument_lookup", {}) if isinstance(context, dict) else {}

    for i, inst in enumerate(names):
        t0 = perf_counter()
        sid = str(SECURITY_ID_MAP.get(inst, "")).strip() if isinstance(SECURITY_ID_MAP, dict) else ""

        kwargs = {
            "access_token": ACCESS_TOKEN,
            "client_id": CLIENT_ID,
            "instrument_name": inst,
            "exchange_segment": EXCHANGE_SEGMENT,
            "timezone_name": "Asia/Kolkata",
            "max_rows": MAX_ROWS,
            "request_timeout": REQUEST_TIMEOUT,
        }
        if SUPPORTS_SECURITY_ID and sid:
            kwargs["security_id"] = sid
            kwargs["resolved_name"] = inst

        result = get_15min_buildup(**kwargs)
        candles = result.get("candles_1m", []) or []
        interval_meta = build_interval_meta_map(candles)

        inst_key = _normalize_inst_key(inst)
        leg_context = context_lookup.get(inst_key) or context_lookup.get(str(inst).strip().upper()) or {}
        if leg_context:
            result["chain_context"] = leg_context

        ltp = float(candles[-1]["close"]) if candles else 0.0
        open_px = float(candles[0]["open"]) if candles else 0.0
        chg = ltp - open_px
        chg_pct = (chg / open_px * 100.0) if open_px else 0.0
        sign = "+" if chg >= 0 else ""

        print(f"{result['instrument']['resolved_name'].upper()}   {ltp:.2f}   {sign}{chg:.2f} ({sign}{chg_pct:.2f}%)")

        spot_line = "Spot Price : SKIPPED (fast mode)"
        if isinstance(context, dict):
            spot = context.get("spot", {}) or {}
            spot_price = spot.get("price")
            spot_time = spot.get("time") or context.get("as_of")
            if spot_price is not None:
                spot_line = f"Spot Price : {float(spot_price):.2f} ({spot_time})"
        print(spot_line)
        print()

        print("Interval       Trading Zone   Direction(1-5m) Conf  Signal    Price Range                OI OI Chg (%)      Fresh   Square-Off     Volume")
        print("-" * 138)
        for row in result.get("rows", []):
            price_range = f"{row['price_range'][0]:.2f} - {row['price_range'][1]:.2f}"
            meta = interval_meta.get(row["interval"], {"direction": "NA", "confidence": "LOW"})
            direction = meta["direction"]
            confidence = meta["confidence"]
            signal = signal_from_confluence(direction, row["trading_zone"])
            signal_colored = color_signal_text(signal)

            print(
                f"{row['interval']:<14} "
                f"{row['trading_zone']:<14} "
                f"{direction:<14} "
                f"{confidence:<5} "
                f"{signal_colored} "
                f"{price_range:<18} "
                f"{inr(row['oi']):>10} "
                f"{row['oi_change_pct']:>9.2f}% "
                f"{inr(row['fresh']):>10} "
                f"{inr(row['square_off']):>12} "
                f"{inr(row['volume']):>10}"
            )

        # print(f"\nLatency: {perf_counter() - t0:.2f}s")
        if i < len(names) - 1:
            print("\n" + "=" * 138 + "\n")





In [ ]:
built_up()


NIFTY 13 APR 23950 CALL   240.00   +36.00 (+17.65%)
Spot Price : 23980.85 (2026-04-08 14:39)

Interval       Trading Zone   Direction(1-5m) Conf  Signal    Price Range                OI OI Chg (%)      Fresh   Square-Off     Volume
------------------------------------------------------------------------------------------------------------------------------------------
14:30 - 14:45  Short Covering DOWN           MED   -         233.15 - 244.70     24,03,180     -4.19%          0     1,05,170  14,72,900
14:15 - 14:30  Short Covering DOWN           MED   -         222.50 - 239.95     25,08,350     -7.04%          0     1,89,930  26,03,705
14:00 - 14:15  Short Buildup  SIDE           LOW   -         219.20 - 236.50     26,98,280      8.85%   2,23,275        3,900  24,68,700
13:45 - 14:00  Long Buildup   UP             HIGH  LONG      223.45 - 237.50     24,78,905      4.62%   1,51,905       42,510  25,03,930
13:30 - 13:45  Long Unwinding DOWN           HIGH  SHORT     230.15 - 256.65     

In [6]:
reno

{'instrument': {'name_input': 'NIFTY 07 APR 23850 PUT',
  'resolved_name': 'NIFTY 07 APR 23850 PUT',
  'security_id': '54807',
  'exchange_segment': 'NSE_FNO',
  'instrument': 'OPTIDX'},
 'range': {'from': '2026-04-08 09:15:00',
  'to': '2026-04-08 14:39:47',
  'timezone': 'Asia/Kolkata'},
 'rows': [{'interval': '14:30 - 14:45',
   'trading_zone': 'Neutral',
   'price_range': [147.6, 156.1],
   'oi': 2278250,
   'oi_change_pct': -0.5,
   'fresh': 19565,
   'square_off': 31070,
   'volume': 447070},
  {'interval': '14:15 - 14:30',
   'trading_zone': 'Long Unwinding',
   'price_range': [151.0, 168.65],
   'oi': 2289755,
   'oi_change_pct': -1.15,
   'fresh': 24050,
   'square_off': 50765,
   'volume': 971165},
  {'interval': '14:00 - 14:15',
   'trading_zone': 'Short Covering',
   'price_range': [152.35, 170.05],
   'oi': 2316470,
   'oi_change_pct': -2.72,
   'fresh': 14495,
   'square_off': 79300,
   'volume': 1220700},
  {'interval': '13:45 - 14:00',
   'trading_zone': 'Long Unwinding